## §0 Setup — tất cả tham số tunable ở đây

In [ ]:
import gc
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

# ===== Paths =====
DATA_DIR = pathlib.Path("data")
OUT_DIR = pathlib.Path("cleaned-data")
TMP_PARQUET = pathlib.Path("/tmp/ratings_intermediate.parquet")

# ===== K-core thresholds =====
USER_MIN = 10              # mỗi user phải có >= USER_MIN ratings
ANIME_MIN = 20             # mỗi anime phải được >= ANIME_MIN ratings
KCORE_MAX_ITER = 20

# ===== Spam/bot thresholds =====
SPAM_COUNT = 2000          # bot dương: rated_count > SPAM_COUNT
SPAM_MEAN = 9.0            #            AND mean_score > SPAM_MEAN
CONST_COUNT = 500          # constant rater: rated_count > CONST_COUNT
CONST_STD = 0.3            #                 AND score_std < CONST_STD
WATCHED_MAX = 5000         # physical-impossibility cap (non-PTW count)
NAN_RATER_MIN_WATCHED = 1000   # watched cao + không rate gì = nghi scraper bug / sock-list

# ===== Other flags =====
DROP_UNKNOWN_STATUS = True
DEDUP_KEEP = "last"        # last theo file order; chỉ 6 cặp duplicate nên impact ~0

# ===== Columns to drop =====
DETAILS_DROP_COLS = ["title_japanese", "url", "image_url", "end_date", "year", "season", "producers",
                     "explicit_genres", "licensors", "streaming"]
PROFILES_DROP_COLS = ["location", "watching", "completed", "on_hold",
                      "dropped", "plan_to_watch"]
RATINGS_DROP_COLS = ["is_rewatching", "num_watched_episodes"]

# ===== Missing helper =====
# 5 cột list-string lưu dạng "['a','b']"; empty = "[]" chứ không phải NaN.
# Muốn "% missing" trung thực phải coi "[]" như NaN cho các cột này.
LIST_COLS_AS_NULL = {"genres", "studios", "themes", "demographics", "producers"}

def missing_pct(df):
    """% missing per column. List cols: NaN ∪ '[]'. Còn lại: pandas .isna()."""
    out = {}
    for col in df.columns:
        s = df[col]
        if col in LIST_COLS_AS_NULL:
            out[col] = (s.isna() | s.eq("[]")).mean() * 100
        else:
            out[col] = s.isna().mean() * 100
    return pd.Series(out).round(2)

# ===== Setup =====
OUT_DIR.mkdir(exist_ok=True)
if TMP_PARQUET.exists():
    TMP_PARQUET.unlink()
    print(f"Cleaned stale intermediate parquet: {TMP_PARQUET}")

# Dict tích lũy snapshot trước/sau để vẽ §7
before, after = {}, {}

print("Setup OK.")
print(f"  K-core: USER_MIN={USER_MIN}, ANIME_MIN={ANIME_MIN}")
print(f"  Spam:   ({SPAM_COUNT},{SPAM_MEAN}) | Constant: ({CONST_COUNT},{CONST_STD})")
print(f"  Caps:   watched≤{WATCHED_MAX} | nan-rater watched>{NAN_RATER_MIN_WATCHED}")

## §1 Load `details` & `profiles`, drop weak columns

Hai file nhỏ (~17–19 MB) → pandas eager. Tính before-snapshot luôn để dùng vẽ §7.

In [ ]:
details_raw = pd.read_csv(DATA_DIR / "details.csv")
profiles_raw = pd.read_csv(DATA_DIR / "profiles.csv")

print(f"details_raw: {details_raw.shape}")
print(f"profiles_raw: {profiles_raw.shape}")

# Before-snapshot cho §7
before["n_anime_details"] = len(details_raw)
before["n_users_profiles"] = len(profiles_raw)
before["details_score"] = details_raw["score"].dropna().values
before["details_members"] = details_raw["members"].dropna().values
before["details_null_pct"] = missing_pct(details_raw)
before["profiles_null_pct"] = missing_pct(profiles_raw)  # profiles không có list col → giống .isna()

# Drop weak cols
details_df = details_raw.drop(columns=DETAILS_DROP_COLS)
profiles_df = profiles_raw.drop(columns=PROFILES_DROP_COLS)

# Sets cho filter ratings ở §3
valid_anime_ids = set(details_raw["mal_id"].dropna().astype(int).tolist())
valid_usernames = set(profiles_raw["username"].dropna().tolist())

print(f"\nvalid_anime_ids: {len(valid_anime_ids):,}")
print(f"valid_usernames: {len(valid_usernames):,}")
print(f"\ndetails kept ({len(details_df.columns)} cols): {list(details_df.columns)}")
print(f"profiles kept ({len(profiles_df.columns)} cols): {list(profiles_df.columns)}")

del details_raw, profiles_raw
gc.collect()

## §3 Streaming clean ratings → parquet intermediate

Áp các filter rẻ trên 4.2 GB `ratings.csv` (1 pass streaming):
- Drop `status="unknown"`
- Drop orphan (anime/user không trong details/profiles)
- Dedup `(username, anime_id)` keep last theo file order
- Sink → parquet để các bước sau re-read nhanh hơn CSV nhiều lần

Trước khi sink: 1 cell streaming aggregation lấy before-snapshot cho §7.

In [ ]:
# Before-snapshot từ raw ratings.csv (streaming, không load full)
print("Computing before-snapshot from raw ratings.csv (streaming)...")

before["n_ratings"] = int(
    pl.scan_csv(DATA_DIR / "ratings.csv")
      .select(pl.len().alias("total"))
      .collect(engine="streaming")[0, "total"]
)

before["status_dist"] = (
    pl.scan_csv(DATA_DIR / "ratings.csv")
      .group_by("status")
      .agg(pl.len().alias("count"))
      .collect(engine="streaming")
      .to_pandas()
      .set_index("status")["count"]
)

before["ratings_score"] = (
    pl.scan_csv(DATA_DIR / "ratings.csv")
      .filter(pl.col("score") > 0)
      .group_by("score")
      .agg(pl.len().alias("count"))
      .sort("score")
      .collect(engine="streaming")
      .to_pandas()
)

before["per_user_counts"] = (
    pl.scan_csv(DATA_DIR / "ratings.csv")
      .group_by("username")
      .agg(pl.len().alias("count"))
      .collect(engine="streaming")
      .to_pandas()["count"].values
)
before["per_anime_counts"] = (
    pl.scan_csv(DATA_DIR / "ratings.csv")
      .group_by("anime_id")
      .agg(pl.len().alias("count"))
      .collect(engine="streaming")
      .to_pandas()["count"].values
)
before["n_users_ratings"] = len(before["per_user_counts"])
before["n_anime_ratings"] = len(before["per_anime_counts"])

print(f"  Total ratings:    {before['n_ratings']:,}")
print(f"  Unique users:     {before['n_users_ratings']:,}")
print(f"  Unique anime:     {before['n_anime_ratings']:,}")
print(f"\nStatus dist:\n{before['status_dist']}")

gc.collect()

In [ ]:
# Streaming clean → parquet intermediate
print("Streaming clean ratings → parquet (vài phút)...")

lf = pl.scan_csv(DATA_DIR / "ratings.csv")

if DROP_UNKNOWN_STATUS:
    lf = lf.filter(pl.col("status") != "unknown")

lf = (
    lf.filter(pl.col("username").is_in(list(valid_usernames)))
      .filter(pl.col("anime_id").is_in(list(valid_anime_ids)))
      .unique(subset=["username", "anime_id"], keep=DEDUP_KEEP, maintain_order=True)
)

try:
    lf.sink_parquet(TMP_PARQUET)
except Exception as e:
    print(f"sink_parquet streaming failed ({e}); fallback to collect → write_parquet")
    lf.collect(engine="streaming").write_parquet(TMP_PARQUET)

print(f"  → {TMP_PARQUET}")
print(f"  Size: {TMP_PARQUET.stat().st_size / 1024**2:.1f} MB")
gc.collect()

In [ ]:
# Post-§3 stats (vẫn streaming, output rất nhỏ)
post_total = int(
    pl.scan_parquet(TMP_PARQUET)
      .select(pl.len().alias("total"))
      .collect(engine="streaming")[0, "total"]
)
post_status = (
    pl.scan_parquet(TMP_PARQUET)
      .group_by("status")
      .agg(pl.len().alias("count"))
      .collect(engine="streaming")
      .to_pandas()
      .set_index("status")["count"]
)

print(f"After §3:")
print(f"  Rows: {before['n_ratings']:,} → {post_total:,}  (Δ={post_total - before['n_ratings']:,})")
print(f"\nStatus dist (không còn 'unknown'):\n{post_status}")

## §4 Spam/bot detection

Aggregate per-user stats trên parquet (streaming). Apply 4 rules union:
- **Bot dương:** `rated_count > SPAM_COUNT` AND `mean_score > SPAM_MEAN` — catch user rate rất nhiều với điểm trung bình cao bất thường. Base là `rated_count` (số rows có `score > 0`) để khớp với `mean_score`/`std_score`.
- **Constant rater:** `rated_count > CONST_COUNT` AND `score_std < CONST_STD` — catch mass-1, mass-10, mass-5 đối xứng.
- **Physical impossibility:** `watched_count > WATCHED_MAX` — `watched_count` = số bản ghi có `status != plan_to_watch`. Không ai xem nổi 6000 anime đời thực; rule này catch bot khôn (mean/std realistic) mà 2 rule trên bỏ lọt.
- **NaN-rater:** `watched_count > NAN_RATER_MIN_WATCHED` AND `mean_score` is null — list rất dài nhưng không rate gì, nghi scraper bug / sock-list.

Score stats chỉ tính trên `score > 0` (loại plan_to_watch chưa rate); vì vậy 2 rule đầu cũng dùng `rated_count` làm base, không phải `count` toàn bộ.

In [ ]:
print("Computing per-user stats from parquet (streaming)...")

user_stats = (
    pl.scan_parquet(TMP_PARQUET)
      .group_by("username")
      .agg(
          pl.len().alias("count"),
          (pl.col("status") != "plan_to_watch").sum().alias("watched_count"),
          (pl.col("score") > 0).sum().alias("rated_count"),
          pl.col("score").filter(pl.col("score") > 0).mean().alias("mean_score"),
          pl.col("score").filter(pl.col("score") > 0).std().alias("std_score"),
      )
      .collect(engine="streaming")
)

spam_pos_mask    = (user_stats["rated_count"] > SPAM_COUNT) & (user_stats["mean_score"] > SPAM_MEAN)
const_rater_mask = (user_stats["rated_count"] > CONST_COUNT) & (user_stats["std_score"] < CONST_STD)
physical_mask    = user_stats["watched_count"] > WATCHED_MAX
nan_lister_mask = (
    (user_stats["watched_count"] > NAN_RATER_MIN_WATCHED)
    & user_stats["mean_score"].is_null()
)
bad_mask         = spam_pos_mask | const_rater_mask | physical_mask | nan_lister_mask

bad_users = set(user_stats.filter(bad_mask)["username"].to_list())

print(f"  Bot dương          (rated>{SPAM_COUNT} & mean>{SPAM_MEAN}):  {int(spam_pos_mask.sum()):,}")
print(f"  Constant rater     (rated>{CONST_COUNT} & std<{CONST_STD}):    {int(const_rater_mask.sum()):,}")
print(f"  Physical impossible (watched>{WATCHED_MAX}):     {int(physical_mask.sum()):,}")
print(f"  NaN-rater         (watched>{NAN_RATER_MIN_WATCHED} & no score):  {int(nan_lister_mask.sum()):,}")
print(f"  Bad union:                                {len(bad_users):,}")

masks = {
    "spam_pos":    spam_pos_mask,
    "const_rater": const_rater_mask,
    "physical":    physical_mask,
    "nan_lister":  nan_lister_mask,
}
print("\nRule contribution (exclusive = catch only bởi rule này):")
for name, m in masks.items():
    others = None
    for n2, m2 in masks.items():
        if n2 == name:
            continue
        others = m2 if others is None else (others | m2)
    excl = int((m & ~others).sum())
    total = int(m.sum())
    print(f"  {name:<12s}  total={total:>5,}  exclusive={excl:>5,}")

print("\nTop 20 bad users (sorted by watched_count):")
print(user_stats.filter(bad_mask)
                .sort("watched_count", descending=True)
                .head(20).to_pandas())

del user_stats
gc.collect()

## §5 Iterative k-core

Materialize chỉ 2 cột `(username, anime_id)` từ parquet (vài trăm MB) sau khi loại `bad_users`. Iterate trong memory: drop user < `USER_MIN` → drop anime < `ANIME_MIN` → lặp đến hết drop.

In [ ]:
print("Materializing (username, anime_id) pairs...")

pairs = (
    pl.scan_parquet(TMP_PARQUET)
      .filter(~pl.col("username").is_in(list(bad_users)))
      .select("username", "anime_id")
      .collect(engine="streaming")
)
print(f"  Initial after dropping bad_users: {len(pairs):,}")

prev_size = -1
for it in range(KCORE_MAX_ITER):
    user_counts = pairs.group_by("username").agg(pl.len().alias("c"))
    valid_users = user_counts.filter(pl.col("c") >= USER_MIN)["username"]
    pairs = pairs.filter(pl.col("username").is_in(valid_users.implode()))

    anime_counts = pairs.group_by("anime_id").agg(pl.len().alias("c"))
    valid_anime = anime_counts.filter(pl.col("c") >= ANIME_MIN)["anime_id"]
    pairs = pairs.filter(pl.col("anime_id").is_in(valid_anime.implode()))

    size = len(pairs)
    print(f"  Iter {it+1:2d}: users={len(valid_users):>7,}  anime={len(valid_anime):>6,}  pairs={size:>11,}")
    if size == prev_size:
        print(f"  → Converged at iter {it+1}.")
        break
    prev_size = size

users_final = set(pairs["username"].unique().to_list())
anime_final = set(pairs["anime_id"].unique().to_list())
n_pairs_final = len(pairs)

print(f"\nFinal: |users|={len(users_final):,}, |anime|={len(anime_final):,}, |pairs|={n_pairs_final:,}")

del pairs, user_counts, anime_counts, valid_users, valid_anime
gc.collect()

## §6 Write final outputs → `cleaned-data/`

3 file CSV overwrite mặc định. Cleanup parquet intermediate cuối cùng.

In [ ]:
# ratings.csv (streaming, có thể ra file lớn ~1-2 GB)
print("Writing cleaned-data/ratings.csv (streaming)...")
(
    pl.scan_parquet(TMP_PARQUET)
      .filter(pl.col("username").is_in(list(users_final)))
      .filter(pl.col("anime_id").is_in(list(anime_final)))
      .drop(*RATINGS_DROP_COLS)
      .sink_csv(OUT_DIR / "ratings.csv")
)
print(f"  → {OUT_DIR / 'ratings.csv'}")

# details.csv
print("Writing cleaned-data/details.csv ...")
details_clean = details_df[details_df["mal_id"].isin(anime_final)].copy()
details_clean.to_csv(OUT_DIR / "details.csv", index=False)
print(f"  → {OUT_DIR / 'details.csv'} ({len(details_clean):,} rows × {len(details_clean.columns)} cols)")

# profiles.csv
print("Writing cleaned-data/profiles.csv ...")
profiles_clean = profiles_df[profiles_df["username"].isin(users_final)].copy()
profiles_clean.to_csv(OUT_DIR / "profiles.csv", index=False)
print(f"  → {OUT_DIR / 'profiles.csv'} ({len(profiles_clean):,} rows × {len(profiles_clean.columns)} cols)")

# Cleanup parquet
if TMP_PARQUET.exists():
    TMP_PARQUET.unlink()
    print(f"\nCleaned up intermediate: {TMP_PARQUET}")

del details_clean, profiles_clean
gc.collect()

## §7 Biểu đồ so sánh before/after

Tính after-stats từ cleaned files (pandas cho details/profiles; polars streaming cho ratings).

In [ ]:
# After-stats từ cleaned-data/
print("Computing after-stats from cleaned-data/ ...")

details_after = pd.read_csv(OUT_DIR / "details.csv")
profiles_after = pd.read_csv(OUT_DIR / "profiles.csv")

after["n_anime_details"] = len(details_after)
after["n_users_profiles"] = len(profiles_after)
after["details_score"] = details_after["score"].dropna().values
after["details_members"] = details_after["members"].dropna().values
after["details_null_pct"] = missing_pct(details_after)
after["profiles_null_pct"] = missing_pct(profiles_after)

after["n_ratings"] = int(
    pl.scan_csv(OUT_DIR / "ratings.csv")
      .select(pl.len().alias("total"))
      .collect(engine="streaming")[0, "total"]
)
after["status_dist"] = (
    pl.scan_csv(OUT_DIR / "ratings.csv")
      .group_by("status")
      .agg(pl.len().alias("count"))
      .collect(engine="streaming")
      .to_pandas()
      .set_index("status")["count"]
)
after["ratings_score"] = (
    pl.scan_csv(OUT_DIR / "ratings.csv")
      .filter(pl.col("score") > 0)
      .group_by("score")
      .agg(pl.len().alias("count"))
      .sort("score")
      .collect(engine="streaming")
      .to_pandas()
)
after["per_user_counts"] = (
    pl.scan_csv(OUT_DIR / "ratings.csv")
      .group_by("username")
      .agg(pl.len().alias("count"))
      .collect(engine="streaming")
      .to_pandas()["count"].values
)
after["per_anime_counts"] = (
    pl.scan_csv(OUT_DIR / "ratings.csv")
      .group_by("anime_id")
      .agg(pl.len().alias("count"))
      .collect(engine="streaming")
      .to_pandas()["count"].values
)
after["n_users_ratings"] = len(after["per_user_counts"])
after["n_anime_ratings"] = len(after["per_anime_counts"])

del details_after, profiles_after
gc.collect()
print("After-stats OK.")

In [ ]:
# Stats summary table
def sparsity(n_users, n_anime, n_ratings):
    if n_users == 0 or n_anime == 0:
        return float("nan")
    return 1 - n_ratings / (n_users * n_anime)

metrics = [
    ("n_users (profiles)",   before["n_users_profiles"],  after["n_users_profiles"],  "int"),
    ("n_users (in ratings)", before["n_users_ratings"],   after["n_users_ratings"],   "int"),
    ("n_anime (details)",    before["n_anime_details"],   after["n_anime_details"],   "int"),
    ("n_anime (in ratings)", before["n_anime_ratings"],   after["n_anime_ratings"],   "int"),
    ("n_ratings",            before["n_ratings"],         after["n_ratings"],         "int"),
    ("sparsity",
        sparsity(before["n_users_ratings"], before["n_anime_ratings"], before["n_ratings"]),
        sparsity(after["n_users_ratings"],  after["n_anime_ratings"],  after["n_ratings"]),
        "rate"),
    ("mean ratings/user",    before["per_user_counts"].mean(),  after["per_user_counts"].mean(),  "float"),
    ("mean ratings/anime",   before["per_anime_counts"].mean(), after["per_anime_counts"].mean(), "float"),
]

def fmt(val, kind):
    if pd.isna(val):    return "—"
    if kind == "int":   return f"{int(val):,}"
    if kind == "rate":  return f"{val * 100:.4f}%"
    if kind == "float": return f"{val:,.2f}"
    return str(val)

def fmt_delta(b, a):
    if not b or pd.isna(b):
        return "—"
    return f"{(a - b) / b * 100:+.2f}%"

W = (22, 16, 16, 12)
print(f"{'metric':<{W[0]}} {'before':>{W[1]}} {'after':>{W[2]}} {'delta':>{W[3]}}")
print("-" * (sum(W) + 3))
for name, b, a, kind in metrics:
    print(f"{name:<{W[0]}} {fmt(b, kind):>{W[1]}} {fmt(a, kind):>{W[2]}} {fmt_delta(b, a):>{W[3]}}")

# Giữ làm DataFrame nếu cần truy cập programmatic sau
stats_df = pd.DataFrame(
    [(n, b, a) for n, b, a, _ in metrics],
    columns=["metric", "before", "after"]
).set_index("metric")

## §8 Summary

Cleaning xong. 3 file đã ghi vào `cleaned-data/`:

- **`cleaned-data/details.csv`** — anime metadata sau drop weak cols (`title_japanese, url, image_url, explicit_genres, licensors, streaming`) + filter theo k-core.
- **`cleaned-data/profiles.csv`** — user metadata sau drop weak cols (`watching, completed, on_hold, dropped, plan_to_watch`) + filter theo k-core. Giữ `username, gender, birthday, location, joined`.
- **`cleaned-data/ratings.csv`** — interaction sau drop `unknown` status + orphan + dedup + spam + k-core. Giữ `username, anime_id, status, score`.

Spam/bot rules áp dụng: **4** (bot dương, constant rater, physical-impossibility cap `watched > WATCHED_MAX`, NaN-rater).

Audit chi tiết tỉ lệ missing thực tế của list cols + sanity check post-cleaning: xem **§9**.

Sẵn sàng cho training pipeline (two-tower retriever + ranker).

> 💡 Muốn thử threshold khác? Chỉnh hằng số trong §0 → **Kernel → Restart & Run All**.

In [ ]:
# §9a Audit empty-list trong list-string columns
# "0% null" ở plot 3 cũ là false signal cho 5 cột này vì pandas không coi "[]" là NaN.
# (Plot 3 nay đã được sửa qua missing_pct() để gộp NaN ∪ '[]'.)
# Cell này đếm cụ thể từng phần: % NaN riêng vs % empty-list riêng.

LIST_COLS = ["genres", "studios", "themes", "demographics"]

details_check = pd.read_csv(OUT_DIR / "details.csv", usecols=LIST_COLS)
n = len(details_check)

rows = []
for col in LIST_COLS:
    s = details_check[col]
    pct_nan = s.isna().mean() * 100
    pct_empty = s.eq("[]").mean() * 100          # empty-list string
    pct_missing = pct_nan + pct_empty            # thực sự thiếu thông tin
    rows.append((col, pct_nan, pct_empty, pct_missing))

audit_df = pd.DataFrame(rows, columns=["column", "% NaN", "% '[]'", "% missing (NaN+[])"])
print(f"Total rows: {n:,}\n")
print(audit_df.to_string(index=False, float_format=lambda x: f"{x:6.2f}"))

del details_check
gc.collect()


In [ ]:
# §9b Cross-tab: empty-list có tương quan với type/popularity không?
import ast

LIST_COLS = ["genres", "studios", "themes", "demographics"]
df = pd.read_csv(OUT_DIR / "details.csv",
                 usecols=LIST_COLS + ["type", "members", "score"])

for col in LIST_COLS:
    df[f"{col}_empty"] = df[col].fillna("[]").eq("[]")

# Bảng 1: % empty theo type
print("=== % empty theo anime type ===")
pivot = (df.groupby("type")[[f"{c}_empty" for c in LIST_COLS]]
           .mean().mul(100).round(1))
pivot["n_anime"] = df.groupby("type").size()
print(pivot.sort_values("n_anime", ascending=False))

# Bảng 2: members trung vị giữa empty vs non-empty (kiểm tra obscurity)
print("\n=== members trung vị: empty vs non-empty ===")
for col in LIST_COLS:
    med_empty = df.loc[df[f"{col}_empty"], "members"].median()
    med_full  = df.loc[~df[f"{col}_empty"], "members"].median()
    ratio = med_full / med_empty if med_empty else float("inf")
    print(f"  {col:<13} empty={med_empty:>8,.0f}  non-empty={med_full:>8,.0f}  ratio={ratio:.1f}x")

del df; gc.collect()


In [ ]:
# §9c Post-cleaning sanity check — watched_count + rated_count + verify 4 rule
# Sau khi áp 4 bot rules ở §4, không user nào còn hold rule sau khi đã write CSV.
print("Per-user stats từ cleaned-data/ratings.csv (streaming)...")

user_stats_after = (
    pl.scan_csv(OUT_DIR / "ratings.csv")
      .group_by("username")
      .agg(
          (pl.col("status") != "plan_to_watch").sum().alias("watched_count"),
          (pl.col("score") > 0).sum().alias("rated_count"),
          pl.col("score").filter(pl.col("score") > 0).mean().alias("mean_score"),
          pl.col("score").filter(pl.col("score") > 0).std().alias("std_score"),
      )
      .collect(engine="streaming")
)

print("\nPercentiles of watched_count:")
for p in [50, 90, 95, 99, 99.5, 99.9, 99.99]:
    v = user_stats_after["watched_count"].quantile(p / 100)
    print(f"  p{p:>5}: {int(v):>6,}")
print(f"  max  : {int(user_stats_after['watched_count'].max()):>6,}  (expect <= {WATCHED_MAX:,})")

print("\nPercentiles of rated_count:")
for p in [50, 90, 95, 99, 99.5, 99.9, 99.99]:
    v = user_stats_after["rated_count"].quantile(p / 100)
    print(f"  p{p:>5}: {int(v):>6,}")
print(f"  max  : {int(user_stats_after['rated_count'].max()):>6,}")

print("\nTop 30 by watched_count (post-cleaning):")
print(
    user_stats_after.sort("watched_count", descending=True)
                    .head(30)
                    .to_pandas()
                    .to_string(index=False)
)

for thr in [3000, 5000, 6000, 8000, 10000]:
    n_over = int((user_stats_after["watched_count"] > thr).sum())
    print(f"  users with watched > {thr:>5,}: {n_over:,}")

# Hậu kiểm 4 rule bot: expect 0 each
spam_pos_after = int(((user_stats_after["rated_count"] > SPAM_COUNT)
                      & (user_stats_after["mean_score"] > SPAM_MEAN)).sum())
const_rater_after = int(((user_stats_after["rated_count"] > CONST_COUNT)
                         & (user_stats_after["std_score"] < CONST_STD)).sum())
physical_after = int((user_stats_after["watched_count"] > WATCHED_MAX).sum())
nan_lister_after = int(((user_stats_after["watched_count"] > NAN_RATER_MIN_WATCHED)
                        & user_stats_after["mean_score"].is_null()).sum())
print("\nPost-cleaning rule verification (expect 0 each):")
print(f"  spam_pos    (rated>{SPAM_COUNT} & mean>{SPAM_MEAN}): {spam_pos_after}")
print(f"  const_rater (rated>{CONST_COUNT} & std<{CONST_STD}):   {const_rater_after}")
print(f"  physical    (watched>{WATCHED_MAX}):                  {physical_after}")
print(f"  nan_lister  (watched>{NAN_RATER_MIN_WATCHED} & no score):  {nan_lister_after}")

del user_stats_after
gc.collect()